In [4]:
import requests
import pandas as pd
import time
import wbgapi as wb
import yfinance as yf


In [5]:
#!pip install wbgapi yfinance

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------------
# 1. Load annual US debt (same as before)
# ------------------------------------------------------------------
debt_raw = pd.read_excel("../data/debt2gdp (1).xlsx", header=0, skiprows=[1,2], usecols="A,BJ:BX")
debt_raw = debt_raw.rename(columns={debt_raw.columns[0]: "country_name"})
us = debt_raw[debt_raw["country_name"].str.strip() == "United States"]
years = [c for c in debt_raw.columns if str(c).isdigit()]
us_debt = us.melt(id_vars="country_name", value_vars=years,
                  var_name="year", value_name="debt")
us_debt["year"] = pd.to_numeric(us_debt["year"])
us_debt["date"] = pd.to_datetime(us_debt["year"].astype(int).astype(str) + "-01-01")
us_debt = us_debt[["date", "debt"]].dropna(subset=["debt"]).set_index("date")
us_debt = us_debt[~us_debt.index.duplicated()]    # remove any duplicate years

# ------------------------------------------------------------------
# 2. Resample to monthly and simple interpolations (linear & pchip)
# ------------------------------------------------------------------
min_date = us_debt.index.min().to_period('M').to_timestamp('S')
max_date = us_debt.index.max().to_period('M').to_timestamp('S')
full_monthly_index = pd.date_range(start=min_date, end=max_date, freq='MS')

monthly = pd.DataFrame(index=full_monthly_index)
monthly['debt'] = us_debt['debt'].reindex(full_monthly_index)
monthly['debt'] = pd.to_numeric(monthly['debt'], errors='coerce')

monthly["debt_linear"] = monthly["debt"].interpolate(method="linear", limit_direction="both")
monthly["debt_pchip"]  = monthly["debt"].interpolate(method="pchip", limit_direction="both")

# ------------------------------------------------------------------
# 3. Add Denton quarterly interpolation (using quarterly GDP)
# ------------------------------------------------------------------
try:
    # Load quarterly GDP sheet – assumes it exists
    gdp_q_raw = pd.read_excel("../data/GDP (1).xlsx", sheet_name='quarterly', header=None)
    gdp_q_raw = gdp_q_raw.drop(1, axis=0).reset_index(drop=True)
    gdp_q_raw.columns = gdp_q_raw.iloc[0]
    gdp_q_raw = gdp_q_raw.drop(0).reset_index(drop=True)
    date_col = gdp_q_raw.columns[0]
    gdp_q_raw = gdp_q_raw.rename(columns={date_col: "date"})
    gdp_q_raw = gdp_q_raw[gdp_q_raw["date"].astype(str).str.contains("Q", na=False)]
    gdp_q_raw["quarter"] = pd.PeriodIndex(gdp_q_raw["date"].str.strip(), freq='Q')
    gdp_q_raw = gdp_q_raw[["quarter", "United States"]].copy()
    gdp_q_raw.columns = ["quarter", "gdp"]
    gdp_q_raw["gdp"] = pd.to_numeric(gdp_q_raw["gdp"], errors="coerce")
    gdp_q = gdp_q_raw.set_index("quarter").sort_index()["gdp"]

    # Restrict to debt years
    first_year = us_debt.index.min().year
    last_year  = us_debt.index.max().year
    gdp_q = gdp_q[(gdp_q.index.year >= first_year) & (gdp_q.index.year <= last_year)]

    # ----- Step A: Annual debt → quarterly using Denton (mean of quarterly GDP) -----
    quarters = gdp_q.index
    years_from_q = quarters.year
    # Map annual debt to each quarter (same value for all quarters of a year)
    annual_debt_by_q = pd.Series(
        years_from_q.map(lambda y: us_debt.loc[pd.Timestamp(f"{y}-01-01")]['debt']),
        index=quarters
    )
    # Annual mean GDP from quarterly means (for mean-preserving Denton)
    annual_gdp_mean = gdp_q.groupby(years_from_q).mean()
    # Match each quarter to its annual mean GDP
    quarterly_gdp_mean = annual_gdp_mean.loc[years_from_q].values
    # Denton proportional disaggregation (mean-preserving):
    # quarterly_debt / quarterly_gdp = annual_debt / annual_gdp_mean
    quarterly_debt = annual_debt_by_q * (gdp_q / quarterly_gdp_mean)

    # ----- Step B: Quarterly → monthly using pchip interpolation -----
    # Place quarterly values at the first month of each quarter
    q_start = quarters.to_timestamp(how='S')
    # Create a separate monthly series for the Denton result, then merge back
    quarterly_aligned = pd.Series(quarterly_debt.values, index=q_start, name="debt_denton")
    # Reindex to the full monthly grid used by the other series
    monthly_denton = quarterly_aligned.reindex(monthly.index)
    # Interpolate (pchip) within the gap between quarter starts
    monthly["debt_denton"] = monthly_denton.interpolate(method="pchip", limit_direction="both")
    denton_available = True
except Exception as e:
    print(f"Denton quarterly interpolation not available: {e}")
    denton_available = False

# ------------------------------------------------------------------
# 4. Show a subset of all three methods
# ------------------------------------------------------------------
print("Monthly US debt (first 15 rows):")
cols = ["debt_linear", "debt_pchip"]
if denton_available:
    cols.append("debt_denton")
print(monthly[cols].head(15))

# ------------------------------------------------------------------
# 5. Plot all on the same scale with visible lines
# ------------------------------------------------------------------
plt.figure(figsize=(14,6))
plt.plot(monthly.index, monthly["debt_linear"], label="Linear", linewidth=2, alpha=0.8, linestyle='--')
plt.plot(monthly.index, monthly["debt_pchip"], label="Pchip", linewidth=2, alpha=0.8, linestyle='-.')
if denton_available:
    plt.plot(monthly.index, monthly["debt_denton"], label="Denton (quarterly GDP)", linewidth=2, alpha=0.8)
plt.scatter(us_debt.index, us_debt["debt"], color='black', s=50, label='Annual observations', zorder=5)
plt.title("US Debt-to-GDP: Monthly Interpolation Methods")
plt.ylabel("Debt/GDP (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1. DENTON PROPORTIONAL TEMPORAL DISAGGREGATION HELPER
# ------------------------------------------------------------------
def denton_proportional(target_series, indicator_series, freq_ratio):
    """
    Disaggregate a low-frequency series to a higher frequency using
    the proportional Denton method.

    Parameters
    ----------
    target_series : pd.Series with DatetimeIndex (low frequency, e.g., annual Jan-1)
    indicator_series : pd.Series with DatetimeIndex (high frequency)
    freq_ratio : int, number of high-frequency periods per low-frequency period
                 (e.g., 12 for annual→monthly, 3 for quarterly→monthly)

    Returns
    -------
    pd.Series with the same index as indicator_series containing the interpolated values.
    """
    # Align indicator to a DataFrame to compute period averages
    df = indicator_series.to_frame('indicator')
    # Determine which low-frequency period each high-frequency observation belongs to
    if freq_ratio == 12:
        df['period'] = df.index.year
    elif freq_ratio == 4:   # quarterly
        df['period'] = df.index.to_period('Q')
    elif freq_ratio == 3:   # quarterly → monthly
        df['period'] = df.index.to_period('Q')
    else:
        raise ValueError("freq_ratio must be 12 (annual->monthly), 4 or 3 (quarterly->monthly)")

    # Average of the indicator for each low-frequency period
    period_avg = df.groupby('period')['indicator'].transform('mean')

    # Proportional scaling
    df['share'] = df['indicator'] / period_avg

    # Map the low-frequency target value to each high-frequency period
    # Build a mapping from period to target value
    if freq_ratio == 12:
        target_values = target_series.reindex(pd.date_range(target_series.index.min(),
                                                            target_series.index.max(), freq='YS'))
        period_to_target = {d.year: v for d, v in target_values.items()}
        df['target'] = df['period'].map(period_to_target)
    elif freq_ratio in (3, 4):
        # For quarterly target, index is the first month of each quarter
        # We assume target_series index is quarter start (e.g., 2010-01-01)
        # Build a mapping from quarter period to value
        q_period_idx = target_series.index.to_period('Q')
        period_to_target = dict(zip(q_period_idx, target_series.values))
        df['target'] = df['period'].map(period_to_target)

    df['disagg'] = df['target'] * df['share']
    return df['disagg']

# ------------------------------------------------------------------
# 2. LOAD ANNUAL US DEBT
# ------------------------------------------------------------------
debt_raw = pd.read_excel("../data/debt2gdp (1).xlsx", header=0, skiprows=[1,2], usecols="A,BJ:BX")
debt_raw = debt_raw.rename(columns={debt_raw.columns[0]: "country_name"})
us = debt_raw[debt_raw["country_name"].str.strip() == "United States"]
years = [c for c in debt_raw.columns if str(c).isdigit()]
us_debt = us.melt(id_vars="country_name", value_vars=years,
                  var_name="year", value_name="debt")
us_debt["year"] = pd.to_numeric(us_debt["year"])
us_debt["date"] = pd.to_datetime(us_debt["year"].astype(int).astype(str) + "-01-01")
us_debt = us_debt.set_index("date")["debt"].dropna().astype(float)
# Keep only yearly points
us_debt = us_debt[us_debt.index.month == 1]

# ------------------------------------------------------------------
# 3. LOAD QUARTERLY US GDP (assumes sheet 'quarterly', date format 2010Q1)
# ------------------------------------------------------------------
gdp_raw = pd.read_excel("../data/GDP (1).xlsx", sheet_name="quarterly", header=None)
gdp_raw = gdp_raw.drop(1, axis=0).reset_index(drop=True)   # drop empty row
gdp_raw.columns = gdp_raw.iloc[0]
gdp_raw = gdp_raw.drop(0).reset_index(drop=True)
date_col = gdp_raw.columns[0]
gdp_raw = gdp_raw.rename(columns={date_col: "date"})
# Keep only quarterly rows
gdp_raw = gdp_raw[gdp_raw["date"].astype(str).str.contains("Q", na=False)]
# Parse to quarterly Period, then to timestamp (first month of quarter)
gdp_raw["quarter"] = pd.PeriodIndex(gdp_raw["date"].str.strip(), freq='Q')
gdp_raw["date"] = gdp_raw["quarter"].dt.to_timestamp()  # first month of quarter
gdp = gdp_raw[["date", "United States"]].dropna().rename(columns={"United States": "gdp"})
gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")
gdp = gdp.set_index("date").sort_index()["gdp"]

# ------------------------------------------------------------------
# 4. DISAGGREGATE QUARTERLY GDP → MONTHLY GDP (using itself as indicator)
# ------------------------------------------------------------------
# For Denton, we need a monthly indicator. Use a simple linear interpolation
# of the quarterly values (assigned to quarter‑start) as a preliminary indicator.
# Then apply Denton to ensure monthly values sum to the quarterly totals.
# First, create a monthly DatetimeIndex
monthly_range = pd.date_range(start=gdp.index.min(), end=gdp.index.max() + pd.DateOffset(months=2), freq='MS')
# Preliminary monthly indicator from linear interpolation of quarterly data
# (place quarterly values at the first month of each quarter)
gdp_monthly_ind = gdp.reindex(monthly_range).interpolate(method='linear', limit_direction='both')
# Apply Denton to enforce quarterly totals
gdp_quarterly = gdp  # now `gdp` is a Series
gdp_monthly = denton_proportional(gdp_quarterly, gdp_monthly_ind, freq_ratio=3)
gdp_monthly.name = "gdp_monthly"

# ------------------------------------------------------------------
# 5. DISAGGREGATE ANNUAL DEBT → MONTHLY DEBT (using monthly GDP as indicator)
# ------------------------------------------------------------------
# Align the monthly GDP to the debt date range
debt_monthly_ind = gdp_monthly.reindex(pd.date_range(us_debt.index.min(),
                                                     us_debt.index.max() + pd.DateOffset(months=11),
                                                     freq='MS'))
# Fill any missing indicator months (e.g., before first GDP quarter) via forward/backward fill
debt_monthly_ind = debt_monthly_ind.ffill().bfill()
# Apply Denton
debt_monthly = denton_proportional(us_debt, debt_monthly_ind, freq_ratio=12)
debt_monthly.name = "debt_monthly"

# ------------------------------------------------------------------
# 6. COMBINE AND PLOT
# ------------------------------------------------------------------
combined = pd.concat([debt_monthly, gdp_monthly], axis=1)

# Filter combined to start from the earliest date of us_debt to avoid NaNs for debt_monthly
combined = combined[combined.index >= us_debt.index.min()]

print("First 12 months of interpolated series:")
print(combined.head(12))

# Plot
fig, ax1 = plt.subplots(figsize=(12,5))
ax1.plot(combined.index, combined["debt_monthly"], label="Debt/GDP (monthly, Denton)", color="tab:blue")
ax1.scatter(us_debt.index, us_debt.values, color="blue", marker="o", s=50, label="Annual debt obs")
ax1.set_ylabel("Debt/GDP (%)", color="tab:blue")
ax1.tick_params(axis='y', labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(combined.index, combined["gdp_monthly"], label="GDP (monthly, Denton)", color="tab:green", alpha=0.7)
ax2.scatter(gdp.index, gdp.values, color="green", marker="s", s=30, label="Quarterly GDP obs")
ax2.set_ylabel("GDP", color="tab:green")
ax2.tick_params(axis='y', labelcolor="tab:green")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.title("US Debt/GDP and GDP – Monthly via Denton Proportional Disaggregation")
plt.show()

In [ ]:
# ---------- VERIFICATION CODE (add after interpolation) ----------

# 1. Check debt: average of monthly values should equal annual value
debt_monthly_avg = debt_monthly.resample('YS').mean()          # yearly mean
debt_annual_comp = pd.DataFrame({
    'original': us_debt.reindex(debt_monthly_avg.index, method=None),  # us_debt already at Jan-1
    'interpolated_avg': debt_monthly_avg
}).dropna()

debt_annual_comp['difference'] = debt_annual_comp['original'] - debt_annual_comp['interpolated_avg']
print("Debt: Annual mean check (original - interpolated average):")
print(debt_annual_comp)
print("Max absolute difference:", np.max(np.abs(debt_annual_comp['difference'])))

# 2. GDP: mean of monthly values should equal quarterly average (index/rate)
gdp_monthly_q = gdp_monthly.to_frame('gdp_monthly')
gdp_monthly_q['quarter'] = gdp_monthly_q.index.to_period('Q')
quarterly_mean = gdp_monthly_q.groupby('quarter')['gdp_monthly'].mean()
original_quarterly = gdp_quarterly.rename('original_gdp')
original_quarterly.index = original_quarterly.index.to_period('Q')
gdp_quarterly_comp = pd.DataFrame({'mean_of_monthly': quarterly_mean, 'original': original_quarterly}).dropna()
gdp_quarterly_comp['difference'] = gdp_quarterly_comp['original'] - gdp_quarterly_comp['mean_of_monthly']
print("\nGDP: Quarterly mean check (original - mean of monthly):")
print(gdp_quarterly_comp.head(10))
print("Max absolute difference:", np.max(np.abs(gdp_quarterly_comp['difference'])))

In [ ]:
import pandas as pd
import yfinance as yf
import warnings
import builtins

warnings.filterwarnings("ignore")
str = builtins.str                     # fix yfinance shadowing bug

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------
YIELD_PATH = "../data/10y_bond_yields (1).xlsx"
NEWS_PATH = "../data/news_topics (1).csv"
UCDP_PATH = "../data/ucdp.csv"
DEBT_PATH = "../data/debt2gdp (1).xlsx"
START_DATE = "2010-01-01"

ISO_MAP = {
    "United States": "USA", "Canada": "CAN", "Belgium": "BEL", "Switzerland": "CHE",
    "Germany": "DEU", "South Africa": "ZAF", "Netherlands": "NLD", "France": "FRA",
    "United Kingdom": "GBR", "Australia": "AUS", "Euro Area": "EUR", "New Zealand": "NZL",
    "Ireland": "IRL", "Spain": "ESP", "Norway": "NOR", "Sweden": "SWE", "Denmark": "DNK",
    "Italy": "ITA", "Iceland": "ISL", "Finland": "FIN", "Portugal": "PRT", "Brazil": "BRA",
    "Israel": "ISR", "Greece": "GRC", "Hungary": "HUN", "Czech Republic": "CZE",
    "Slovakia": "SVK", "South Korea": "KOR", "Latvia": "LVA", "Poland": "POL",
    "Lithuania": "LTU", "Mexico": "MEX", "Slovenia": "SVN", "Bulgaria": "BGR",
    "Colombia": "COL", "Chile": "CHL", "Romania": "ROU", "Croatia": "HRV", "India": "IND",
    "Costa Rica": "CRI", "Indonesia": "IDN", "China": "CHN", "Estonia": "EST",
}

TOPIC_NAMES = {
    'stock_topic_0':'Competition and Sports', 'stock_topic_1':'Health and Education',
    'stock_topic_2':'Military Conflict', 'stock_topic_3':'Politics',
    'stock_topic_4':'Military Technology', 'stock_topic_5':'National Development',
    'stock_topic_6':'Political Tensions', 'stock_topic_7':'Judiciary and Abuses',
    'stock_topic_8':'Middle East', 'stock_topic_9':'Chinese Politics',
    'stock_topic_10':'Economics', 'stock_topic_11':'Diplomacy',
    'stock_topic_12':'Civilian Life', 'stock_topic_13':'Foreign Policy',
    'stock_topic_14':'Power and Negotiation'
}

# ---------------------------------------------------------------------------
# DATA LOADERS
# ---------------------------------------------------------------------------
def load_bond_yields(path, iso_map):
    headers = pd.read_excel(path, header=1, nrows=0).columns.tolist()
    headers[0] = "date"
    df = pd.read_excel(path, skiprows=29, names=headers)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.melt(id_vars="date", var_name="country_name", value_name="yield_10y")
    df["isocode"] = df["country_name"].map(iso_map)
    df = df.dropna(subset=["isocode", "yield_10y"])
    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp()
    return df.groupby(["isocode", "date"])["yield_10y"].mean().reset_index()

def load_news_ucdp(news_path, ucdp_path):
    news = pd.read_csv(news_path)
    news["date"] = pd.to_datetime(news["period"].astype(str), format="%Y%m")
    ucdp = pd.read_csv(ucdp_path)
    ucdp = ucdp.rename(columns={"iso_a3": "isocode"})
    ucdp["date"] = pd.to_datetime(ucdp["period"].astype(str), format="%Y%m")
    return news, ucdp

def load_imf_debt(path):
    df = pd.read_excel(path, header=0, skiprows=[1,2], usecols="A,BJ:BX")
    df = df.rename(columns={df.columns[0]: "country_name"})
    years = [c for c in df.columns if str(c).isdigit()]
    df = df.melt(id_vars="country_name", value_vars=years, var_name="year", value_name="debt")
    df["date"] = pd.to_datetime(df["year"].astype(str) + "-01-01")
    df["isocode"] = df["country_name"].map(ISO_MAP)
    df["debt"] = pd.to_numeric(df["debt"], errors="coerce") # Ensure debt is numeric
    return df.dropna(subset="isocode")[["isocode", "date", "debt"]]

def load_market_globals():
    symbols = {"^VIX": "vix", "BZ=F": "brent_oil", "HYG": "credit_spread"}
    series_list = []
    for ticker, name in symbols.items():
        data = yf.download(ticker, start=START_DATE, interval="1mo")
        if data.empty:
            continue
        col = ("Adj Close", ticker) if ("Adj Close", ticker) in data.columns else ("Close", ticker)
        s = data[col].resample("MS").mean()
        s.name = name
        series_list.append(s)
    if not series_list:
        return pd.DataFrame(columns=["date", "vix", "brent_oil", "credit_spread"])
    df = pd.concat(series_list, axis=1)
    df["date"] = df.index
    df = df.reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    return df[["date", "vix", "brent_oil", "credit_spread"]]

def build_macro_panel():
    """Proven loader: all 7 WB monthly sheets, maps country names to ISO3."""
    FILES = {
        "../data/CPI (2).xlsx": "inflation",
        "../data/ER (1).xlsx": "fx_rate",
        "../data/Industrial Production (1).xlsx": "ip",
        "../data/Stock Markets (1).xlsx": "stock_index",
        "../data/Terms of Trade.xlsx": "terms_of_trade",
    }
    from functools import reduce
    dfs = []
    for filename, varname in FILES.items():
        xls = pd.ExcelFile(filename)
        sheet = 'monthly' if 'monthly' in xls.sheet_names else 0
        df = pd.read_excel(filename, sheet_name=sheet, header=None)
        # Drop empty second row if present
        df = df.drop(1, axis=0).reset_index(drop=True)
        # First row -> header
        df.columns = df.iloc[0]
        df = df.drop(0).reset_index(drop=True)
        date_col = df.columns[0]
        df = df.rename(columns={date_col: "date"})
        # Keep monthly or annual rows
        date_str = df["date"].astype(str).str.strip()
        monthly = date_str.str.match(r"^\d{4}M\d{2}$")
        annual = date_str.str.match(r"^\d{4}$")
        df = df[monthly | annual].copy()
        # Parse
        df["date"] = pd.to_datetime(date_str.str.replace("M", "-"), format="%Y-%m", errors="coerce")
        df.loc[df["date"].isna(), "date"] = pd.to_datetime(date_str[df["date"].isna()], format="%Y", errors="coerce")
        df = df.dropna(subset=["date"])
        # Melt
        country_cols = [c for c in df.columns if c != "date"]
        df_long = df.melt(id_vars="date", value_vars=country_cols,
                          var_name="country_name", value_name=varname)
        # Clean & map
        df_long["country_name"] = df_long["country_name"].astype(str).str.strip()
        df_long["isocode"] = df_long["country_name"].map(ISO_MAP)
        df_long = df_long.dropna(subset=["isocode", varname])
        dfs.append(df_long[["isocode", "date", varname]])
    macro = reduce(lambda left, right: pd.merge(left, right, on=["isocode", "date"], how="outer"), dfs)
    macro = macro[macro["date"] >= START_DATE].sort_values(["isocode", "date"]).reset_index(drop=True)
    return macro

def interpolate_debt_simple(df, value_col="debt", group_col="isocode", date_col="date"):
    """Interpolate annual debt to monthly using monotone cubic (pchip) per country."""
    # Ensure sorting
    df = df.sort_values([group_col, date_col]).copy()
    # Create a combined group identifier for transformation
    df[value_col] = df.groupby(group_col)[value_col].transform(
        lambda x: x.interpolate(method="pchip", limit_direction="both")
    )
    return df

    # ------------------------------------------------------------------
# DENTON HELPER (same as before, handles both annual→monthly and quarterly→monthly)
# ------------------------------------------------------------------
def denton_proportional(target_series, indicator_series, freq_ratio):
    """
    Disaggregate a low‑frequency series to a higher frequency using
    the proportional Denton method.
    """
    df = indicator_series.to_frame('indicator')
    if freq_ratio == 12:               # annual → monthly
        df['period'] = df.index.year
    elif freq_ratio in (3, 4):         # quarterly → monthly
        df['period'] = df.index.to_period('Q')
    else:
        raise ValueError("freq_ratio must be 12 or 3/4")

    period_avg = df.groupby('period')['indicator'].transform('mean')
    df['share'] = df['indicator'] / period_avg

    # Map low‑frequency target onto each high‑frequency period
    if freq_ratio == 12:
        target_by_period = {d.year: v for d, v in target_series.items()}
        df['target'] = df['period'].map(target_by_period)
    else:
        q_period_idx = target_series.index.to_period('Q')
        target_by_period = dict(zip(q_period_idx, target_series.values))
        df['target'] = df['period'].map(target_by_period)

    df['disagg'] = df['target'] * df['share']
    return df['disagg']

# ------------------------------------------------------------------
# LOAD QUARTERLY GDP (all countries)
# ------------------------------------------------------------------
def load_quarterly_gdp(file_path, iso_map):
    """Load quarterly GDP from WB export (sheet 'quarterly', date format 2010Q1)."""
    gdp_raw = pd.read_excel(file_path, sheet_name='quarterly', header=None)
    gdp_raw = gdp_raw.drop(1, axis=0).reset_index(drop=True)   # drop empty row
    gdp_raw.columns = gdp_raw.iloc[0]
    gdp_raw = gdp_raw.drop(0).reset_index(drop=True)
    date_col = gdp_raw.columns[0]
    gdp_raw = gdp_raw.rename(columns={date_col: "date"})
    gdp_raw = gdp_raw[gdp_raw["date"].astype(str).str.contains("Q", na=False)]
    # Parse to quarter start timestamp
    gdp_raw["quarter"] = pd.PeriodIndex(gdp_raw["date"].str.strip(), freq='Q')
    gdp_raw["date"] = gdp_raw["quarter"].dt.to_timestamp()   # first month of quarter
    # Melt country columns
    country_cols = [c for c in gdp_raw.columns if c not in ("date", "quarter")]
    gdp_long = gdp_raw.melt(id_vars=["date", "quarter"], value_vars=country_cols,
                            var_name="country_name", value_name="gdp")
    gdp_long["country_name"] = gdp_long["country_name"].str.strip()
    gdp_long["isocode"] = gdp_long["country_name"].map(iso_map)
    gdp_long = gdp_long.dropna(subset=["isocode", "gdp"])
    return gdp_long[["isocode", "date", "gdp"]].sort_values(["isocode", "date"])

# ------------------------------------------------------------------
# DISAGGREGATE QUARTERLY GDP → MONTHLY, THEN DEBT → MONTHLY FOR ALL COUNTRIES
# ------------------------------------------------------------------
def add_denton_interpolations(master, gdp_quarterly_file="../data/GDP (1).xlsx"):
    """
    Incorporate Denton‑disaggregated monthly GDP and monthly debt into the master panel.
    """
    # 1. Load and disaggregate quarterly GDP → monthly GDP per country
    gdp_q = load_quarterly_gdp(gdp_quarterly_file, ISO_MAP)
    # For each country: build monthly range, create rough indicator from linear interpolation,
    # then apply Denton to enforce quarterly means.
    monthly_gdp_list = []
    for iso, group in gdp_q.groupby("isocode"):
        group = group.set_index("date").sort_index()
        # monthly range covering all quarters
        monthly_idx = pd.date_range(start=group.index.min(), end=group.index.max() + pd.DateOffset(months=2), freq='MS')
        # Preliminary indicator (linear interpolation of quarterly values)
        ind = group["gdp"].reindex(monthly_idx).interpolate(method='linear', limit_direction='both')
        # Denton disaggregation
        target = group["gdp"]  # quarterly values on quarter‑start dates
        monthly_gdp = denton_proportional(target, ind, freq_ratio=3)
        monthly_gdp = monthly_gdp.rename("gdp_monthly").to_frame()
        monthly_gdp["isocode"] = iso
        monthly_gdp = monthly_gdp.reset_index().rename(columns={"index": "date"})
        monthly_gdp_list.append(monthly_gdp)
    gdp_monthly = pd.concat(monthly_gdp_list, ignore_index=True)
    gdp_monthly["date"] = pd.to_datetime(gdp_monthly["date"])

    # Merge monthly GDP into master
    master = master.merge(gdp_monthly, on=["isocode", "date"], how="left")

    # 2. Disaggregate annual debt → monthly debt (using monthly GDP as indicator)
    # First, ensure we have annual debt (Jan‑1 values) in the master; we already have 'debt' column.
    # For each country, extract the annual debt series (Jan‑1 rows), then apply Denton.
    monthly_debt_list = []
    for iso, group in master.groupby("isocode"):
        # Annual debt points (Jan‑1)
        annual = group[group["date"].dt.month == 1].set_index("date")["debt"].sort_index().dropna()
        if len(annual) < 2:
            # Not enough points to interpolate; forward‑fill the existing value
            group = group.set_index("date").copy()
            group["debt_monthly"] = group["debt"].ffill()
            monthly_debt_list.append(group.reset_index()[["isocode", "date", "debt_monthly"]])
            continue
        # Monthly indicator (GDP) – use the just‑created gdp_monthly column
        ind = group.set_index("date").sort_index()["gdp_monthly"]
        # Align monthly indicator range to annual debt range
        start = annual.index.min()
        end = annual.index.max() + pd.DateOffset(months=11)
        monthly_idx = pd.date_range(start=start, end=end, freq='MS')
        ind = ind.reindex(monthly_idx).interpolate(method='linear', limit_direction='both').ffill().bfill()
        # Denton
        monthly_debt = denton_proportional(annual, ind, freq_ratio=12)
        monthly_debt = monthly_debt.rename("debt_monthly").to_frame()
        monthly_debt["isocode"] = iso
        monthly_debt = monthly_debt.reset_index().rename(columns={"index": "date"})
        monthly_debt_list.append(monthly_debt)
    debt_monthly_all = pd.concat(monthly_debt_list, ignore_index=True)
    debt_monthly_all["date"] = pd.to_datetime(debt_monthly_all["date"])

    # Merge monthly debt into master
    master = master.merge(debt_monthly_all, on=["isocode", "date"], how="left")
    # Replace the old 'debt' column with the interpolated one
    master["debt"] = master["debt_monthly"].combine_first(master["debt"])
    master = master.drop(columns=["debt_monthly", "gdp_monthly"], errors="ignore")
    return master


# ---------------------------------------------------------------------------
# BUILD MASTER PANEL
# ---------------------------------------------------------------------------
def build_panel():
    yields = load_bond_yields(YIELD_PATH, ISO_MAP)
    news, ucdp = load_news_ucdp(NEWS_PATH, UCDP_PATH)
    debt = load_imf_debt(DEBT_PATH)
    macro = build_macro_panel()
    market = load_market_globals()

    yields = yields[yields["date"] >= START_DATE]

    for df_ in (yields, news, ucdp, debt, macro, market):
        df_["date"] = pd.to_datetime(df_["date"]).dt.to_period("M").dt.to_timestamp()

    master = (yields
              .merge(news, on=["isocode", "date"], how="inner")
              .merge(ucdp[["isocode", "date", "fatalities_UCDP"]], on=["isocode", "date"], how="left")
              .merge(market, on="date", how="left")
              .merge(macro, on=["isocode", "date"], how="left")
              .merge(debt, on=["isocode", "date"], how="left"))

    master = master.sort_values(["isocode", "date"])
    master["debt"] = master.groupby("isocode")["debt"].transform(
        lambda x: x.interpolate(method="cubic"))

    master = add_denton_interpolations(master)
    return master

# ---------------------------------------------------------------------------
# EXECUTION
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    df = build_panel()
    controls = [
        "inflation", "ip", "fx_rate", "vix", "brent_oil",
        "credit_spread", "debt", "fatalities_UCDP"
    ]
    topic_cols = [c for c in df.columns if "topic" in c]
    print(f"Panel: {df.shape[0]} rows \u00d7 {df.shape[1]} columns")
    print(f"Topics found: {len(topic_cols)})")
    print(df[["isocode", "date", "yield_10y"] + controls].head(10))

In [ ]:
# ------------------------------------------------------------------
# TEMPORAL CONTINUITY CHECK
# ------------------------------------------------------------------
from collections import defaultdict

def check_continuity(df, date_col="date", group_col="isocode"):
    """
    For each country, verify:
      - Consecutive months (no gaps in the date index)
      - Missing values per column
    Returns a summary DataFrame.
    """
    df = df.sort_values([group_col, date_col]).copy()
    # Expected monthly frequency
    expected_months = pd.date_range(
        start=df[date_col].min(), end=df[date_col].max(), freq="MS"
    )

    summary = defaultdict(list)
    for iso, grp in df.groupby(group_col):
        grp = grp.set_index(date_col).sort_index()
        # 1. Date gaps: reindex to expected range and see if any months are completely absent
        full_idx = expected_months[(expected_months >= grp.index.min()) & (expected_months <= grp.index.max())]
        missing_months = full_idx.difference(grp.index)
        summary["isocode"].append(iso)
        summary["start"].append(grp.index.min())
        summary["end"].append(grp.index.max())
        summary["expected_months"].append(len(full_idx))
        summary["actual_months"].append(len(grp))
        summary["missing_months"].append(len(missing_months))

        # 2. Missing values per variable (NaN percentage)
        for col in df.columns:
            if col in (group_col, date_col):
                continue
            missing_pct = 100 * grp[col].isna().mean()
            summary.setdefault(f"{col}_missing_pct", []).append(missing_pct)

    report = pd.DataFrame(summary)
    # Show the worst columns (highest average missingness)
    missing_cols = [c for c in report.columns if c.endswith("_missing_pct")]
    avg_missing = report[missing_cols].mean().sort_values(ascending=False)

    print("=== Temporal Continuity Report ===")
    print(f"Countries examined: {len(report)}")
    print(f"Total rows in panel: {len(df)}")
    print("\nTop columns by average missingness (across countries):")
    for col in avg_missing.index[:10]:
        print(f"  {col.replace('_missing_pct','')}: {avg_missing[col]:.1f}%")

    print("\nCountries with missing months (gaps in time index):")
    gap_countries = report[report["missing_months"] > 0]
    if len(gap_countries) == 0:
        print("  None – all countries have complete monthly grids.")
    else:
        for _, row in gap_countries.iterrows():
            print(f"  {row['isocode']}: {row['missing_months']} missing months out of {row['expected_months']}")

    print("\nDetailed report (first 5 countries):")
    display_cols = ["isocode", "start", "end", "expected_months", "actual_months", "missing_months"] + missing_cols[:5]
    print(report[display_cols].head(5).to_string())

    return report

# Run the check
continuity_report = check_continuity(df)

In [ ]:
# Rebuild the expected monthly grid per country and show which months are absent
expected_months = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="MS")

for iso in ["MEX", "CRI", "GRC"]:
    grp = df[df["isocode"] == iso].set_index("date").sort_index()
    full_idx = expected_months[(expected_months >= grp.index.min()) & (expected_months <= grp.index.max())]
    missing = full_idx.difference(grp.index)
    if len(missing):
        print(f"{iso}: {len(missing)} missing months")
        print(missing.tolist())
    else:
        print(f"{iso}: complete")

In [ ]:
# ------------------------------------------------------------------
# HANDLE YIELD GAPS
# ------------------------------------------------------------------
# 1. Drop Mexico – too many missing yields (51 months)
df = df[df["isocode"] != "MEX"].copy()

# 2. For CRI (6 missing) and GRC (1 missing), linearly interpolate yields
#    We reindex each country to a full monthly grid, fill the few missing yields,
#    then forward/backward-fill any remaining NaN in other columns.
def fix_small_gaps(group):
    expected = pd.date_range(group["date"].min(), group["date"].max(), freq="MS")
    group = group.set_index("date").reindex(expected)
    group["isocode"] = group["isocode"].ffill().bfill()   # fill the grouping column
    # Interpolate yield linearly (only for the few missing months)
    group["yield_10y"] = group["yield_10y"].interpolate(method="linear", limit=3)
    # For other variables, forward-fill then backward-fill (no interpolation)
    group = group.fillna(method="ffill").fillna(method="bfill")
    return group.reset_index().rename(columns={"index":"date"})

# Apply only to countries with small gaps (we know CRI and GRC)
small_gap_countries = ["CRI", "GRC"]
for iso in small_gap_countries:
    mask = df["isocode"] == iso
    df_fixed = fix_small_gaps(df[mask])
    df = df[~mask]  # remove old rows
    df = pd.concat([df, df_fixed], ignore_index=True)

df = df.sort_values(["isocode", "date"]).reset_index(drop=True)

# 3. Quick re‑check (optional)
missing_check = df.groupby("isocode")["yield_10y"].apply(lambda x: x.isna().sum())
print("Remaining yield missing values per country:")
print(missing_check[missing_check > 0])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Choose a representative basket (adjust as you like)
basket = ["USA", "DEU", "GBR", "JPN", "AUS", "BRA", "KOR", "ZAF"]

plt.figure(figsize=(14,6))
for iso in basket:
    if iso in df["isocode"].values:
        sub = df[df["isocode"] == iso].set_index("date").sort_index()
        plt.plot(sub.index, sub["yield_10y"], label=iso, linewidth=1.5)

plt.title("10‑Year Government Bond Yields (selected countries)", fontsize=14)
plt.ylabel("Yield (%)")
plt.legend(loc="upper right", ncol=4, fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------
# 0. Common settings
# ------------------------------------------------------------------
MAX_LAG = 24
min_obs = MAX_LAG + 24           # 48 months – enough for the Granger test

# Rename topic columns to human‑readable names
rename_dict = {k: v for k, v in TOPIC_NAMES.items() if k in df.columns}
df = df.rename(columns=rename_dict)
topic_cols = [rename_dict.get(c, c) for c in topic_cols]   # original 'topic_cols' variable updated

# ------------------------------------------------------------------
# 1. Determine common eligible countries for both scans
# ------------------------------------------------------------------
country_counts = df.groupby('isocode').size()
eligible_isos = country_counts[country_counts >= min_obs].index
print(f"Total countries with at least {min_obs} months: {len(eligible_isos)}")

# ------------------------------------------------------------------
# 2. News scan (all 15 topics)
# ------------------------------------------------------------------
news_topics = list(TOPIC_NAMES.values())   # already human‑readable after rename
news_topics_existing = [t for t in news_topics if t in df.columns]
print(f"Testing {len(news_topics_existing)} news topics: {news_topics_existing}")

def granger_news_scan(df, topics, iso_list, max_lag=MAX_LAG):
    results = []
    print(f"Scanning {len(iso_list)} countries...")
    for iso in iso_list:
        c_df = df[df['isocode'] == iso].sort_values('date').copy()
        c_df['dy'] = c_df['yield_10y'].diff()
        for topic in topics:
            if topic not in c_df.columns:
                continue
            c_df['dx'] = c_df[topic].diff()
            data = c_df[['dy', 'dx']].dropna()
            if len(data) <= max_lag:
                continue
            try:
                gc_res = grangercausalitytests(data, maxlag=max_lag, verbose=False)
                for lag in range(1, max_lag + 1):
                    p_val = gc_res[lag][0]['params_ftest'][1]
                    results.append({
                        'isocode': iso, 'topic': topic, 'lag': lag, 'p_value': p_val
                    })
            except Exception:
                continue
    return pd.DataFrame(results)

news_results = granger_news_scan(df, news_topics_existing, eligible_isos)

# Aggregate
news_results['significant'] = news_results['p_value'] < 0.05
news_leaderboard = news_results.groupby(['topic', 'lag'])['significant'].mean().reset_index()
news_leaderboard['pct'] = news_leaderboard['significant'] * 100
pivot_news = news_leaderboard.pivot(index='topic', columns='lag', values='pct')

# Heatmap
plt.figure(figsize=(16, 10))
sns.heatmap(pivot_news, annot=True, cmap='viridis', fmt=".1f",
            linewidths=0.3, cbar_kws={'label': '% Countries Significant (p<0.05)'})
plt.title("News Topics Lead‑Lag Map: ΔTopic → Δ10Y Yield (24‑Month Horizon)", fontsize=14)
plt.xlabel("Lag (months)")
plt.ylabel("News Topic")
plt.tight_layout()
plt.show()

print("\n--- Top 15 News Topic Lead‑Lag Signals ---")
print(news_leaderboard.sort_values('pct', ascending=False).head(15)[['topic', 'lag', 'pct']].to_string(index=False))

In [ ]:
# ------------------------------------------------------------------
# Macro controls: all available (including GDP)
# ------------------------------------------------------------------
macro_controls_list = [
     'inflation', 'ip', 'fx_rate', 'stock_index','reserves',
    'vix', 'brent_oil', 'credit_spread', 'debt', 'fatalities_UCDP'
]
# Keep only those present in the renamed DataFrame
macro_controls_existing = [c for c in macro_controls_list if c in df.columns]
print(f"Testing {len(macro_controls_existing)} macro/market variables: {macro_controls_existing}")

def granger_macro_scan(df, controls, iso_list, max_lag=MAX_LAG):
    results = []
    print(f"Scanning {len(iso_list)} countries...")
    for iso in iso_list:
        c_df = df[df['isocode'] == iso].sort_values('date').copy()
        c_df['dy'] = c_df['yield_10y'].diff()
        for var in controls:
            if var not in c_df.columns:
                continue
            c_df['dx'] = c_df[var].diff()
            data = c_df[['dy', 'dx']].dropna()
            if len(data) <= max_lag:
                continue
            try:
                gc_res = grangercausalitytests(data, maxlag=max_lag, verbose=False)
                for lag in range(1, max_lag + 1):
                    p_val = gc_res[lag][0]['params_ftest'][1]
                    results.append({
                        'isocode': iso, 'variable': var, 'lag': lag, 'p_value': p_val
                    })
            except Exception:
                continue
    return pd.DataFrame(results)

macro_results = granger_macro_scan(df, macro_controls_existing, eligible_isos)

# Aggregate
macro_results['significant'] = macro_results['p_value'] < 0.05
macro_leaderboard = macro_results.groupby(['variable', 'lag'])['significant'].mean().reset_index()
macro_leaderboard['pct'] = macro_leaderboard['significant'] * 100
pivot_macro = macro_leaderboard.pivot(index='variable', columns='lag', values='pct')

# Heatmap
plt.figure(figsize=(18, 12))
sns.heatmap(pivot_macro, annot=True, cmap='magma', fmt=".1f",
            linewidths=0.3, cbar_kws={'label': '% Countries Significant (p<0.05)'})
plt.title("Macro/Market Controls Lead‑Lag Map: ΔVariable → Δ10Y Yield (24‑Month Horizon)", fontsize=14)
plt.xlabel("Lag (months)")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

print("\n--- Top 15 Macro/Market Lead‑Lag Signals ---")
print(macro_leaderboard.sort_values('pct', ascending=False).head(15)[['variable', 'lag', 'pct']].to_string(index=False))

In [ ]:
def compute_chunked_significance(leaderboard, var_col='variable', lag_col='lag', pct_col='pct'):
    """
    Returns:
      - overall_avg: average % significance per variable (all lags)
      - chunked: pivot table with variables as rows and 4‑month lag chunks as columns
    """
    # Overall average
    overall_avg = leaderboard.groupby(var_col)[pct_col].mean().sort_values(ascending=False).reset_index()

    # Create 4‑month bins: 1‑4, 5‑8, ..., 21‑24
    bins = list(range(1, 25, 4))   # [1, 5, 9, 13, 17, 21]
    labels = [f"{b}-{b+3}" for b in bins]
    leaderboard['chunk'] = pd.cut(leaderboard[lag_col], bins=bins+[25],
                                  right=False, labels=labels)
    chunked = leaderboard.groupby([var_col, 'chunk'])[pct_col].mean().unstack(fill_value=0)
    # Ensure all expected chunks are present
    for lbl in labels:
        if lbl not in chunked.columns:
            chunked[lbl] = 0.0
    chunked = chunked[labels]  # order columns

    return overall_avg, chunked

# ----------------------------------------------------
# 1. News topics
# ----------------------------------------------------
news_overall, news_chunked = compute_chunked_significance(
    news_leaderboard, var_col='topic', lag_col='lag', pct_col='pct'
)
print("=== Overall average significance – News topics ===")
print(news_overall.to_string(index=False))

print("\n=== 4‑month chunked significance – News topics ===")
print(news_chunked.to_string())

# Heatmap for news chunked
plt.figure(figsize=(12, 10))
sns.heatmap(news_chunked, annot=True, cmap='viridis', fmt=".1f",
            cbar_kws={'label': 'Avg % Significant (p<0.05)'})
plt.title("News Topics: Short‑Run vs Long‑Run Significance (4‑month chunks)")
plt.ylabel("Topic")
plt.xlabel("Lag chunk (months)")
plt.tight_layout()
plt.show()

# ----------------------------------------------------
# 2. Macro / Market controls
# ----------------------------------------------------
macro_overall, macro_chunked = compute_chunked_significance(
    macro_leaderboard, var_col='variable', lag_col='lag', pct_col='pct'
)
print("=== Overall average significance – Macro/Market variables ===")
print(macro_overall.to_string(index=False))

print("\n=== 4‑month chunked significance – Macro/Market variables ===")
print(macro_chunked.to_string())

plt.figure(figsize=(12, 8))
sns.heatmap(macro_chunked, annot=True, cmap='viridis', fmt=".1f",
            cbar_kws={'label': 'Avg % Significant (p<0.05)'})
plt.title("Macro/Market: Short‑Run vs Long‑Run Significance (4‑month chunks)")
plt.ylabel("Variable")
plt.xlabel("Lag chunk (months)")
plt.tight_layout()
plt.show()